In [1]:
# Install all packages for this notebook (run once per environment / new kernel)
%pip install -q jupyter ipykernel langgraph langchain-core langchain-openai typing-extensions

Note: you may need to restart the kernel to use updated packages.


# LangGraph for beginners

**What you will learn:** core ideas (state, nodes, edges), how they map to **agent patterns** (router, ReAct, reflection), and runnable **OpenAI** examples.

**Further reading (official & community):**
- [LangGraph docs](https://langchain-ai.github.io/langgraph/) — concepts and API reference  
- [LangGraph tutorials (LangChain blog)](https://blog.langchain.dev/) — patterns and releases  
- GitHub: `langchain-ai/langgraph` — notebooks under `examples/`

> Replace the dummy API key in the setup cell with your real key before running LLM cells.

## 1. Big picture: graph + state

| Concept | Plain English |
|--------|----------------|
| **State** | A shared Python object (usually a `TypedDict`) that every step can **read** and **update**. Think “the notebook the whole workflow shares.” |
| **Node** | A **function** that receives state and returns **updates** to state (a `dict` of fields to merge in). One unit of work: call an LLM, call a tool, parse JSON, etc. |
| **Edge** | “After node A finishes, always go to node B.” Fixed routing. |
| **Conditional edge** | “After node A, **look at state** (or a small router function) and **choose** the next node.” This is how you build **routers**, **ReAct loops**, and **reflection** branches. |
| **START / END** | Special markers: where the run begins and where it stops. |

**Mental model:** `StateGraph(your_state_schema)` → `add_node` → `add_edge` / `add_conditional_edges` → `compile()` → `invoke()`.

### Common "node" flavors (all are just callables + optional reducers)

| Kind | Role |
|------|------|
| **Plain function node** | `def step(state): return {"field": value}` — most examples in this notebook. |
| **LLM node** | Same, but inside you call `llm.invoke(...)` or `llm.bind_tools(...).invoke(...)`. |
| **`ToolNode` (prebuilt)** | Runs tool calls from an `AIMessage`; use with a **conditional edge** to loop back to the LLM. |
| **`create_react_agent`** | Prebuilt **subgraph** that wires LLM ↔ tools for you (ReAct). |

`START` and `END` are not nodes; they mark **entry** and **exit** of the graph.

## 2. Setup: imports and API key

Paste your OpenAI key over the dummy value.

In [2]:
import os
from typing import Literal, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

# Prefer env var (recommended). Fallback is a placeholder — set OPENAI_API_KEY for real runs.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-dummy-replace-with-your-openai-key")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Model used in all examples (adjust if you use a different tier)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

---
## Example A — Linear graph: one LLM “node”

**Pattern:** fixed path `START → greet → END`. No branching.

**Shows:** custom `TypedDict` state, a node that calls the LLM, `add_edge`.

In [3]:
class GreetState(TypedDict):
    topic: str
    reply: str


def greet_node(state: GreetState) -> dict:
    """A node is just a function: input state → return partial state updates."""
    msg = llm.invoke(
        [
            SystemMessage(
                content="You are a friendly tutor. Reply in 2 short sentences."
            ),
            HumanMessage(content=f"Explain briefly: {state['topic']}"),
        ]
    )
    return {"reply": msg.content}


builder = StateGraph(GreetState)
builder.add_node("greet", greet_node)
builder.add_edge(START, "greet")
builder.add_edge("greet", END)

linear_app = builder.compile()
out = linear_app.invoke({"topic": "what is a LangGraph node", "reply": ""})
print(out["reply"])

A LangGraph node represents a specific entity or concept within a graph structure used in natural language processing. It helps in organizing and connecting information to facilitate understanding and analysis of language data.


---
## Example B — Router pattern: conditional edges

**Pattern:** classify the user request, then route to different “experts” (here: `math` vs `chitchat`).

**Shows:** `add_conditional_edges` + a small routing function returning a **literal** next step name.

In [4]:
class RouterState(TypedDict):
    question: str
    route: str
    answer: str


def classify(state: RouterState) -> dict:
    text = state["question"].lower()
    if any(w in text for w in ("+", "plus", "times", "multiply", "what is", "calculate")):
        return {"route": "math"}
    return {"route": "chitchat"}


def math_bot(state: RouterState) -> dict:
    r = llm.invoke(
        [HumanMessage(content=f"Solve or explain briefly (show reasoning): {state['question']}")]
    )
    return {"answer": r.content}


def chitchat_bot(state: RouterState) -> dict:
    r = llm.invoke(
        [HumanMessage(content=f"Reply helpfully and briefly: {state['question']}")]
    )
    return {"answer": r.content}


def pick_route(state: RouterState) -> Literal["math", "chitchat"]:
    return state["route"]  # must match keys in the mapping below


rb = StateGraph(RouterState)
rb.add_node("classify", classify)
rb.add_node("math", math_bot)
rb.add_node("chitchat", chitchat_bot)
rb.add_edge(START, "classify")
rb.add_conditional_edges(
    "classify",
    pick_route,
    {"math": "math", "chitchat": "chitchat"},
)
rb.add_edge("math", END)
rb.add_edge("chitchat", END)

router_app = rb.compile()
print(router_app.invoke({"question": "What is 12 * 7?", "route": "", "answer": ""}))
print("---")
print(router_app.invoke({"question": "Say hi in one sentence", "route": "", "answer": ""}))

{'question': 'What is 12 * 7?', 'route': 'math', 'answer': 'To solve \\( 12 \\times 7 \\), you can use basic multiplication.\n\n1. Break it down: \n   - \\( 12 \\times 7 \\) can be thought of as \\( 12 \\) added to itself \\( 7 \\) times.\n   - Alternatively, you can use the distributive property: \\( 12 \\times 7 = (10 + 2) \\times 7 = 10 \\times 7 + 2 \\times 7 \\).\n\n2. Calculate each part:\n   - \\( 10 \\times 7 = 70 \\)\n   - \\( 2 \\times 7 = 14 \\)\n\n3. Add the results together:\n   - \\( 70 + 14 = 84 \\)\n\nThus, \\( 12 \\times 7 = 84 \\).'}
---
{'question': 'Say hi in one sentence', 'route': 'chitchat', 'answer': 'Hi there! How can I assist you today?'}


---
## Example C — ReAct-style loop: LLM + tools (`ToolNode`)

**Pattern:** the model may **call tools**; we run tools in a `ToolNode`, then send results back to the model until it stops calling tools.

**Shows:** `MessagesState`, tool definitions, `ToolNode`, conditional edge **after** the LLM (tools vs end).

In [5]:
from langchain_core.tools import tool
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode


@tool
def fake_weather(city: str) -> str:
    """Get pretend weather for a city (demo only)."""
    return f"Weather in {city}: 22°C, partly cloudy (demo data)."


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


tools = [fake_weather, add_numbers]
llm_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)


def agent_node(state: MessagesState) -> dict:
    resp = llm_tools.invoke(state["messages"])
    return {"messages": [resp]}


def should_use_tools(state: MessagesState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


react_b = StateGraph(MessagesState)
react_b.add_node("agent", agent_node)
react_b.add_node("tools", tool_node)
react_b.add_edge(START, "agent")
react_b.add_conditional_edges("agent", should_use_tools, {"tools": "tools", END: END})
react_b.add_edge("tools", "agent")

react_app = react_b.compile()
result = react_app.invoke(
    {
        "messages": [
            HumanMessage(
                content="What is 41 + 9? Then tell me the weather in Paris (use tools)."
            )
        ]
    }
)
for m in result["messages"]:
    print(m.type, ":", getattr(m, "content", m))

human : What is 41 + 9? Then tell me the weather in Paris (use tools).
ai : 
tool : 50
tool : Weather in Paris: 22°C, partly cloudy (demo data).
ai : The result of 41 + 9 is 50. 

As for the weather in Paris, it is currently 22°C and partly cloudy (demo data).


---
## Example D — Reflection: draft → critique → revise (loop)

**Pattern:** generate a draft, ask the model to critique, then either **finish** or **revise** once more.

**Shows:** multi-step reasoning, conditional edge on **quality** (here: iteration count + simple critique keyword).

In [6]:
class ReflectState(TypedDict):
    topic: str
    draft: str
    critique: str


def write_draft(state: ReflectState) -> dict:
    r = llm.invoke(
        [HumanMessage(content=f"Write a 3-sentence answer about: {state['topic']}")]
    )
    return {"draft": r.content}


def critique_draft(state: ReflectState) -> dict:
    r = llm.invoke(
        [
            HumanMessage(
                content=(
                    f"Critique this answer briefly (bullets). Say IMPROVE if it needs work.\n\n"
                    f"{state['draft']}"
                )
            )
        ]
    )
    return {"critique": r.content}


def revise(state: ReflectState) -> dict:
    r = llm.invoke(
        [
            HumanMessage(
                content=(
                    f"Improve the draft using the critique. Keep 3 sentences.\n"
                    f"Draft:\n{state['draft']}\n\nCritique:\n{state['critique']}"
                )
            )
        ]
    )
    return {"draft": r.content}


def after_critique(state: ReflectState) -> Literal["revise", "__end__"]:
    # Demo rule: if the critic says IMPROVE, do one revision step, then stop.
    if "IMPROVE" in state["critique"].upper():
        return "revise"
    return END


fb = StateGraph(ReflectState)
fb.add_node("draft", write_draft)
fb.add_node("critique", critique_draft)
fb.add_node("revise", revise)
fb.add_edge(START, "draft")
fb.add_edge("draft", "critique")
fb.add_conditional_edges("critique", after_critique, {"revise": "revise", END: END})
fb.add_edge("revise", END)

reflect_app = fb.compile()
final_state = reflect_app.invoke(
    {"topic": "Why use graphs for agents?", "draft": "", "critique": ""}
)
print("Final draft:\n", final_state["draft"])
print("\nCritique:\n", final_state["critique"])

Final draft:
 Graphs are essential tools for artificial intelligence agents, as they provide a structured framework for representing complex relationships and interactions within data. For instance, in a recommendation system, agents use graphs to analyze user preferences and item similarities, uncovering patterns that optimize decision-making and enhance learning from interconnected data points. Furthermore, graphs facilitate the visualization of these relationships, enabling agents to effectively communicate insights and collaborate with other systems or stakeholders in real-world applications, such as social network analysis or supply chain management.

Critique:
 - **Clarity**: The answer is generally clear but could benefit from more specific examples of how agents use graphs.
- **Depth**: Lacks depth in explaining the types of graphs or specific applications relevant to agents.
- **Terminology**: The term "agents" could be more clearly defined to avoid ambiguity.
- **Structure**:

---
## Example E — Prebuilt ReAct agent (`create_react_agent`)

**Pattern:** same ReAct idea (reason → act → observe), packaged as one helper for common chat + tools workflows.

**Shows:** minimal code path when you do not need a fully custom graph layout.

In [7]:
from langgraph.prebuilt import create_react_agent


@tool
def word_count(text: str) -> int:
    """Count words in a string."""
    return len(text.split())


prebuilt = create_react_agent(llm, [word_count])
out = prebuilt.invoke(
    {"messages": [HumanMessage(content="How many words in: LangGraph makes agents easier?")]}
)
print(out["messages"][-1].content)

C:\Users\SSD\AppData\Local\Temp\ipykernel_19608\1289595232.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  prebuilt = create_react_agent(llm, [word_count])


The phrase "LangGraph makes agents easier?" contains 4 words.


## Recap

- **State** = shared data; **nodes** = functions that update it; **edges** = fixed next step; **conditional edges** = dynamic next step.
- **Router** = classify then `add_conditional_edges`.
- **ReAct** = LLM + tools in a loop (`ToolNode` + route on tool calls, or `create_react_agent`).
- **Reflection** = explicit draft/critique/revise nodes with a loop or conditional exit.

Next steps: add **memory** (`checkpointer`), **human approval** nodes, or **subgraphs** for bigger systems — all in the [LangGraph documentation](https://langchain-ai.github.io/langgraph/).